# SSB Material Property Pipeline v2 — CorrectedFixes the errors found in the earlier pipeline. Sources per property, all recorded in `_src_*` output columns.| Property | Source ||---|---|| Formation energy | Crystalformer (fallback: CHGNet) || Energy above hull (Ehull) | **Materials Project phase diagram** (needs `MP_API_KEY`, Cell 3) || Band gap | Crystalformer (fallback: ALIGNN, else N/A) || Bulk modulus, Shear modulus | Crystalformer (fallback: ALIGNN, else CHGNet finite-strain) || Young's modulus, Poisson's ratio | formula on bulk+shear modulus || Fracture toughness | Irwin-Griffith formula, gamma_s = 0.069 x G (calibrated to LLZO) || Electronic conductivity | Arrhenius formula on band gap || Thermal stability | formula on the **MP-based** Ehull (not Crystalformer's JARVIS-hull Ehull) || Volume change | CHGNet relaxation || Electrochemical window | **Materials Project grand-potential phase diagram**, Li chemical-potential sweep (needs `MP_API_KEY`) |**PASTE YOUR MP_API_KEY IN CELL 3, LINE 2** (`MP_API_KEY = "..."`). Get one free at https://next-gen.materialsproject.org/api .**Crystalformer is NOT plug-and-play.** Unlike ALIGNN, it has no pip-installable pretrained checkpoint loader. To actually use it:1. `git clone https://github.com/omron-sinicx/crystalframer` (the maintained follow-up repo)2. Build/run their Docker image (see their README)3. Manually download pretrained weights from the Google Drive link in their repo4. Point `CRYSTALFORMER_WEIGHTS_DIR` (Cell 3) at the unzipped weightsOnly band gap and formation energy checkpoints are confirmed available for direct download as of writing — bulk/shear modulus and Ehull checkpoints are reported in the paper but not confirmed downloadable. **Cell 6 (`crystalformer_predict`) is a stub** you fill in once you have the model loaded; until then, every property routes through its fallback automatically, and `_src_*` will say so honestly rather than pretending Crystalformer ran.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Cell 1 — install
!pip install -q chgnet pymatgen mp-api ase ipywidgets torch_geometric dgl alignn

In [ ]:
# Cell 2 — imports
import os, io, json, contextlib, warnings
import numpy as np
import pandas as pd
from pymatgen.core import Structure, Composition
warnings.filterwarnings("ignore")
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
MP_API_KEY = ""
CRYSTALFORMER_REPO_DIR = "/content/drive/My Drive/CrystalFormer/crystalframer"
CRYSTALFORMER_WEIGHTS_DIR = "/content/drive/My Drive/CrystalFormer/crystalframer_weights/crystalframer_weights"
CIF_DIR = "uploaded_cifs/cif_files"       # folder of .cif files to run the pipeline on
OUTPUT_CSV = "predictions_v2.csv"

GAMMA_PER_GPA_SHEAR = 0.069          # fracture-toughness surface-energy scaling, calibrated to LLZO
EHULL_STABLE = 0.025                 # eV/atom cutoff: <= this = "stable"
EHULL_METASTABLE = 0.080             # eV/atom cutoff: <= this = "metastable"

if not MP_API_KEY:
    print("WARNING: MP_API_KEY is empty. Ehull, thermal stability and electrochemical window "
          "will all be reported as N/A until you paste a key above.")
if not CRYSTALFORMER_WEIGHTS_DIR or not os.path.isdir(CRYSTALFORMER_WEIGHTS_DIR):
    print("NOTE: CRYSTALFORMER_WEIGHTS_DIR not set/found. Crystalformer-based properties "
          "will use their fallback path instead (see _src_* columns in the output).")

In [ ]:
import os

print('Contents of CRYSTALFORMER_REPO_DIR:')
for root, dirs, files in os.walk(CRYSTALFORMER_REPO_DIR):
    level = root.replace(CRYSTALFORMER_REPO_DIR, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')

print('\nContents of CRYSTALFORMER_WEIGHTS_DIR:')
for root, dirs, files in os.walk(CRYSTALFORMER_WEIGHTS_DIR):
    level = root.replace(CRYSTALFORMER_WEIGHTS_DIR, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')

Contents of CRYSTALFORMER_REPO_DIR:
crystalframer/
    LICENSE
    README.md
    demo.sh
    demo.py
    init_datasets.py
    train.py
    train_jarvis.sh
    train_mp.sh
    train_jarvis_time.sh
    utils.py
    train_oqmd.sh
    torch_scatter.py
    __pycache__/
        utils.cpython-313.pyc
        torch_scatter.cpython-313.pyc
    .git/
        description
        packed-refs
        HEAD
        index
        config
        logs/
            HEAD
            refs/
                remotes/
                    origin/
                        HEAD
                heads/
                    main
        refs/
            remotes/
                origin/
                    HEAD
            heads/
                main
        hooks/
            applypatch-msg.sample
            commit-msg.sample
            fsmonitor-watchman.sample
            post-update.sample
            pre-applypatch.sample
            pre-commit.sample
            pre-merge-commit.sample
            pre-push.sam

In [ ]:
# Cell 4 — load CHGNet (used for: volume change, and as a fallback for formation energy / moduli)
from chgnet.model import CHGNet, StructOptimizer
chgnet_model = CHGNet.load()
relaxer = StructOptimizer(model=chgnet_model)
print("CHGNet ready.")

CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on cuda
CHGNet will run on cuda
CHGNet ready.


In [ ]:
# Cell 5 — (optional) ALIGNN availability check — fallback path for band gap / moduli / Ehull
DGL_AVAILABLE, DGL_ERROR = False, None
# Fix for 'ImportError: cannot import name 'Mapping' from 'collections' in Python 3.10+
import collections
if not hasattr(collections, 'Mapping'):
    import collections.abc
    collections.Mapping = collections.abc.Mapping
# Fix for 'ImportError: cannot import name 'Iterable' from 'collections' in Python 3.10+
if not hasattr(collections, 'Iterable'):
    import collections.abc
    collections.Iterable = collections.abc.Iterable
try:
    import dgl
    dgl.graph(([0], [1]))
    DGL_AVAILABLE = True
except Exception as e:
    DGL_ERROR = f"{type(e).__name__}: {e}"
    print(f"DGL/ALIGNN unavailable -> {DGL_ERROR}. Will skip straight to CHGNet/N/A fallbacks.")

def alignn_predict(model_name, cif_path):
    """Returns (value, None) or (None, reason). Never fails silently."""
    if not DGL_AVAILABLE:
        return None, f"DGL unavailable ({DGL_ERROR})"
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            from alignn.pretrained import get_prediction
            from jarvis.core.atoms import Atoms
            atoms = Atoms.from_cif(cif_path)
            return float(get_prediction(model_name=model_name, atoms=atoms)), None
    except Exception as e:
        return None, f"{model_name} failed: {type(e).__name__}: {e}"

DGL/ALIGNN unavailable -> ImportError: cannot import name 'Iterable' from 'collections' (/usr/lib/python3.13/collections/__init__.py). Will skip straight to CHGNet/N/A fallbacks.


In [ ]:
import sys, pickle, glob as _glob

CRYSTALFORMER_REPO_DIR = "/content/drive/My Drive/CrystalFormer/crystalframer"   # <-- PASTE the path to your cloned crystalframer repo (Cell 3 area)
CRYSTALFORMER_WEIGHTS_DIR = CRYSTALFORMER_WEIGHTS_DIR or "/content/drive/My Drive/CrystalFormer/crystalframer_weights/crystalframer_weights"  # unzipped Release v1.0.1 weights dir

# Map: property name -> (param json filename, checkpoint filename, target_set name, index in `targets` list)
# NOTE: fill these in with the ACTUAL filenames you get after unzipping Release v1.0.1 -- the release
# bundles params + checkpoints per target_set (jarvis__megnet for e_form/bandgap/bulk/shear_modulus,
# jarvis__dft_3d_2021-ehull for ehull). Inspect the unzipped folder and update the paths below.
CRYSTALFORMER_CHECKPOINTS = {
    'formation_energy': dict(param_json="default_mp.json",      ckpt="mp_e_form.ckpt",      target_set="jarvis__megnet",             targets=["e_form"],           idx=0),
    'band_gap':         dict(param_json="default_mp.json",      ckpt="mp_bandgap.ckpt",     target_set="jarvis__megnet",             targets=["bandgap"],          idx=0),
    'bulk_modulus':     dict(param_json="default_mp.json",      ckpt="mp_bulk_modulus.ckpt",        target_set="jarvis__megnet-bulk_modulus", targets=["bulk_modulus"],     idx=0),  # log(GPa) -- exponentiate!
    'shear_modulus':    dict(param_json="default_mp.json",      ckpt="mp_shear_modulus.ckpt",       target_set="jarvis__megnet-shear_modulus",targets=["shear_modulus"],    idx=0),  # log(GPa) -- exponentiate!
    'e_hull_jarvis':    dict(param_json="default_jarvis.json",  ckpt="jarvis_ehull.ckpt",   target_set="jarvis__dft_3d_2021-ehull",   targets=["ehull"],            idx=0),
}

_cf_models = {}   # cache of loaded models, keyed by property name

def _cf_load_one(prop):
    """Loads one Latticeformer checkpoint for one property, following demo.py's exact loading logic."""
    if prop in _cf_models:
        return _cf_models[prop]
    if not CRYSTALFORMER_REPO_DIR or not os.path.isdir(CRYSTALFORMER_REPO_DIR):
        _cf_models[prop] = None
        return None
    cfg = CRYSTALFORMER_CHECKPOINTS[prop]
    param_path = os.path.join(CRYSTALFORMER_REPO_DIR, "params", "crystalframer", cfg['param_json'])
    ckpt_path = os.path.join(CRYSTALFORMER_WEIGHTS_DIR, cfg['ckpt'])
    if not (os.path.exists(param_path) and os.path.exists(ckpt_path)):
        print(f"[Crystalformer] {prop}: checkpoint/param files not found "
              f"({cfg['ckpt']} / {cfg['param_json']} under CRYSTALFORMER_WEIGHTS_DIR) -> will fall back.") # Fixed typo
        _cf_models[prop] = None
        return None
    if CRYSTALFORMER_REPO_DIR not in sys.path:
        sys.path.insert(0, CRYSTALFORMER_REPO_DIR)
    # Their own demo.py does os.chdir() to the repo root before importing anything -- their dataset
    # loader likely resolves "data/..." relative to cwd, not to an absolute path. We mirror that here,
    # and always restore the original cwd afterward so the rest of THIS notebook (e.g. CIF_DIR) is unaffected.
    _orig_cwd = os.getcwd()
    try:
        os.chdir(CRYSTALFORMER_REPO_DIR)
        import torch
        from utils import Params
        from models.latticeformer import Latticeformer
        params = Params(param_path)
        params.pretrained_model = ckpt_path
        params.targets = cfg['targets']
        model = Latticeformer(params)
        with open(ckpt_path, "rb") as f:
            checkpoint = torch.load(f, map_location="cpu")
        state_dict = checkpoint['state_dict']
        target_std = state_dict['target_std']
        target_mean = state_dict['target_mean']
        prefix = "swa_model.module."
        model_dict = {k.replace(prefix, ""): v for k, v in state_dict.items() if k.startswith(prefix)}
        model.load_state_dict(model_dict)
        # de-normalize the final layer in place, exactly as demo.py does
        model.mlp[-1].load_state_dict({
            'weight': model.mlp[-1].weight * target_std[:, None],
            'bias': model.mlp[-1].bias * target_std + target_mean,
        })
        model.eval()
        _cf_models[prop] = (model, cfg)
        print(f"[Crystalformer] {prop}: loaded OK.")
        return _cf_models[prop]
    except Exception as e:
        print(f"[Crystalformer] {prop}: failed to load ({type(e).__name__}: {e}) -> will fall back.")
        _cf_models[prop] = None
        return None
    finally:
        os.chdir(_orig_cwd)

def _cf_structure_to_batch(structure, target_set, cif_path): # Added cif_path parameter
    """
    Wraps ONE pymatgen Structure into the batched torch_geometric input Latticeformer expects,
    following the repo's "Use a custom dataset" pattern (a list of {'structure':..., <target>: dummy}
    dicts, loaded through their own Dataset class so featurization matches training exactly).
    """
    import torch
    from torch_geometric.loader import DataLoader
    from dataloaders.dataset_latticeformer import RegressionDatasetMP_Latticeformer as Dataset

    material_id = os.path.splitext(os.path.basename(cif_path))[0] # Extract material_id
    tmp_name = "_single_predict_tmp"
    split_dir = os.path.join(CRYSTALFORMER_REPO_DIR, "data", tmp_name, "test", "raw")
    os.makedirs(split_dir, exist_ok=True)
    # Clear stale processed cache so new structures are always re-featurized
    proc_dir = os.path.join(CRYSTALFORMER_REPO_DIR, "data", tmp_name, "test", "processed")
    if os.path.isdir(proc_dir):
        import shutil; shutil.rmtree(proc_dir)
    data_list = [{"id": material_id, "structure": structure, "e_form": 0.0, "bandgap": 0.0, # Added 'id' field
                  "bulk_modulus": 0.0, "shear_modulus": 0.0, "ehull": 0.0}]
    with open(os.path.join(split_dir, "raw_data.pkl"), "wb") as f:
        pickle.dump(data_list, f)
    _orig_cwd = os.getcwd()
    try:
        os.chdir(CRYSTALFORMER_REPO_DIR)   # dataset loader likely expects cwd == repo root, see note above
        dataset = Dataset(target_split="test", target_set=tmp_name)
        loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
        batch = next(iter(loader))
        batch.id = material_id # Explicitly add material_id to the batch object
        return batch
    finally:
        os.chdir(_orig_cwd)

def crystalformer_predict(structure, cif_path):
    """
    Returns whatever Crystalformer checkpoints are actually loadable, e.g:
      {'formation_energy': ..., 'band_gap': ..., 'bulk_modulus': ..., 'shear_modulus': ..., 'e_hull_jarvis': ...}
    Missing keys fall through to the ALIGNN/CHGNet/N-A fallbacks already wired into predict_one().
    """
    import torch
    out = {}
    for prop in CRYSTALFORMER_CHECKPOINTS:
        loaded = _cf_load_one(prop)
        if loaded is None:
            continue
        model, cfg = loaded
        try:
            batch = _cf_structure_to_batch(structure, cfg['target_set'], cif_path) # Added cif_path argument
            with torch.no_grad():
                pred = model(batch)[0, cfg['idx']].item()
            if prop in ('bulk_modulus', 'shear_modulus'):
                # MEGNet elastic checkpoints use log10(GPa), not natural log.
                # Keep both representations so the conversion is auditable.
                out[f'{prop}_log10_GPa'] = round(float(pred), 4)
                pred = float(10.0 ** pred)
            out[prop] = round(pred, 4)
        except Exception as e:
            print(f"[Crystalformer] {prop} inference failed for {os.path.basename(cif_path)}: "
                  f"{type(e).__name__}: {e}")
    return out

In [ ]:
print('--- Diagnosing Crystalformer Loading ---')
try:
    # Clear cache to force a fresh load attempt
    _cf_models.clear()
    print("Attempting to load Crystalformer for 'formation_energy'...")
    result = _cf_load_one('formation_energy')
    if result is None:
        print("Crystalformer loading for 'formation_energy' failed. Check detailed messages above.")
    else:
        print("Crystalformer loading for 'formation_energy' succeeded.")
except Exception as e:
    print(f"An unexpected error occurred during Crystalformer debug: {type(e).__name__}: {e}")
print('--- End Diagnosis ---')

--- Diagnosing Crystalformer Loading ---
Attempting to load Crystalformer for 'formation_energy'...
Latticeformer params----
dropout: 0.0
head_num: 8
v_dim: 0
k_dim: 0
pooling: avr
pre_pooling_op: no
scale_grad: 0.0
use_cuda_code: True
relu
Parsed LatticeformerParams:
{'layer_index': -1, 'domain': 'real', 'cos_abs': 0, 'lattice_range': 2, 'minimum_range': True, 'adaptive_cutoff_sigma': -3.5, 'gauss_lb_real': 0.5, 'gauss_lb_reci': 0.5, 'scale_real': [1.4], 'scale_reci': [2.2], 'normalize_gauss': True, 'value_pe_dist_real': 64, 'value_pe_dist_coef': 1.0, 'value_pe_dist_max': -10.0, 'value_pe_dist_wscale': 1.0, 'value_pe_wave_real': 0, 'value_pe_dist_reci': 0, 'value_pe_wave_reci': 0, 'value_pe_angle_real': 64, 'value_pe_angle_coef': 1.0, 'value_pe_angle_wscale': 4.0, 'positive_func_beta': 0.1, 'gauss_state': 'q', 'frame_mode': 'both', 'frame_method': 'max', 'symm_break_noise': 1e-05}
lattice former params
Please install cupy and pytorch-pfn-extras to use the CUDA implementation.
[Crystal

In [ ]:
import os

_mp_cache = {}   # cache per chemical system, so repeated systems don't re-hit the API

def _get_mp_entries(elements, mpr):
    key = tuple(sorted(elements))
    if key not in _mp_cache:
        _mp_cache[key] = mpr.get_entries_in_chemsys(list(key))
    return _mp_cache[key]

def chgnet_total_energy(structure):
    pred = chgnet_model.predict_structure(structure)
    return float(pred['e']) * len(structure)

def compute_ehull_mp(structure, e_total_eV, mpr):
    """Real energy above hull via the MP phase diagram. Returns (ehull_eV_atom, source_str)."""
    if mpr is None:
        return None, "N/A (no MP_API_KEY)"
    try:
        from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry
        elements = sorted({el.symbol for el in structure.composition.elements})
        entries = _get_mp_entries(elements, mpr)
        pd_ = PhaseDiagram(entries)
        ehull = pd_.get_e_above_hull(PDEntry(structure.composition, e_total_eV), allow_negative=True)
        return max(0.0, float(ehull)), "MP phase diagram (CHGNet energy vs MP competing phases)"
    except Exception as e:
        return None, f"MP phase diagram failed: {type(e).__name__}: {e}"

def compute_electrochemical_window_mp(structure, e_total_eV, mpr,
                                       mu_li_range=(-10.0, 5.0), n_steps=141):
    """
    Grand-potential phase diagram sweep vs Li chemical potential (Richards, Miara, Wang, Kim & Ceder 2016).
    Sweeps mu_Li, and at each value checks whether this material is still the stable phase in the
    Li-grand-potential phase diagram built from MP competing phases. The window is the mu_Li range
    (converted to V vs Li/Li+) over which it remains stable.
    Returns (v_reduction, v_oxidation, source_str) — voltages are vs. Li/Li+, so V = -mu_Li.
    """
    if mpr is None:
        return None, None, "N/A (no MP_API_KEY)"
    if not any(el.symbol == 'Li' for el in structure.composition.elements):
        return None, None, "N/A (no Li in structure)"
    try:
        from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry, GrandPotentialPhaseDiagram
        from pymatgen.core import Element
        elements = sorted({el.symbol for el in structure.composition.elements} | {'Li'})
        entries = _get_mp_entries(elements, mpr)
        my_entry = PDEntry(structure.composition, e_total_eV)
        all_entries = list(entries) + [my_entry]

        stable_mus = []
        for mu in np.linspace(mu_li_range[0], mu_li_range[1], n_steps):
            gpd = GrandPotentialPhaseDiagram(all_entries, {Element("Li"): mu})
            gp_entry = [e for e in gpd.all_entries if e.original_entry == my_entry]
            if gp_entry and gp_entry[0] in gpd.stable_entries:
                stable_mus.append(mu)
        if not stable_mus:
            return None, None, "Not found stable at any mu_Li in the swept range -- widen mu_li_range"
        v_ox = -min(stable_mus)    # low mu_Li (Li-poor) = high voltage = oxidation limit
        v_red = -max(stable_mus)   # high mu_Li (Li-rich) = low voltage = reduction limit
        return round(v_red, 3), round(v_ox, 3), "MP grand-potential phase diagram (mu_Li sweep)"
    except Exception as e:
        return None, None, f"Grand-potential sweep failed: {type(e).__name__}: {e}"

mpr = None
if MP_API_KEY:
    from mp_api.client import MPRester
    mpr = MPRester(MP_API_KEY)
    print("Materials Project client ready.")

Materials Project client ready.


In [ ]:
# Cell 8 — derived-formula properties + AIMD thermal decomposition
def youngs_and_poisson(bulk_GPa, shear_GPa):
    K, G = bulk_GPa, shear_GPa
    E = (9*K*G) / (3*K + G)
    nu = (3*K - 2*G) / (2*(3*K + G))
    return round(E, 2), round(max(-0.5, min(0.49, nu)), 3)

def fracture_toughness(youngs_GPa, shear_GPa, poisson):
    gamma_raw = GAMMA_PER_GPA_SHEAR * shear_GPa
    gamma_s = max(0.1, min(5.0, gamma_raw))
    k_ic = np.sqrt(2.0 * youngs_GPa * gamma_s / max(0.01, 1.0 - poisson**2)) * np.sqrt(1e9) / 1e6
    return round(max(0.1, float(k_ic)), 3), gamma_s

def electronic_conductivity(band_gap_eV):
    if band_gap_eV is None:
        return None, "N/A"
    if band_gap_eV < 0.05:
        cls, sigma = "Metallic", 1.0e3
    else:
        cls = "Semiconductor" if band_gap_eV <= 3.0 else "Insulator"
        sigma = 1e3 * np.exp(-band_gap_eV / (2.0 * 0.02585))
    return f"{sigma:.3e}", cls

# ---------- AIMD thermal decomposition (replaces the old Ehull proxy) ----------
def compute_thermal_decomp_aimd(structure,
                                 temp_start=400, temp_end=2000, temp_step=200,
                                 n_equil=200, n_snapshots=100, collect_interval=5,
                                 dt_fs=1.0, lindemann_threshold=0.12):
    """
    Predict thermal decomposition temperature via CHGNet AIMD.

    Procedure:
      1. Build a supercell (>= 32 atoms) so thermal statistics are meaningful.
      2. At each temperature in [temp_start, temp_end, step]:
         a. Set Maxwell-Boltzmann velocities, run Langevin NVT equilibration.
         b. Collect n_snapshots of atomic positions during production run.
         c. Compute Lindemann ratio = RMS displacement from time-averaged
            positions, normalized by nearest-neighbour distance.
         d. If Lindemann > threshold, structure has melted/decomposed at this T.
      3. Return (T_decomp_Celsius, status_string).

    Typical runtime: ~1-3 min per material on GPU.
    """
    from ase.md.langevin import Langevin
    from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
    from ase import units
    from pymatgen.io.ase import AseAtomsAdaptor

    try:
        # --- Supercell ---
        sc = structure.copy()
        if len(sc) < 32:
            factor = int(np.ceil((32 / len(sc)) ** (1/3)))
            sc.make_supercell([factor, factor, factor])

        atoms = AseAtomsAdaptor.get_atoms(sc)

        # --- Calculator ---
        from chgnet.model import CHGNetCalculator
        atoms.calc = CHGNetCalculator(model=chgnet_model)

        # --- Nearest-neighbour distance (reference scale) ---
        dists = atoms.get_all_distances(mic=True)
        np.fill_diagonal(dists, np.inf)
        d_nn = float(np.min(dists))

        temps = list(range(temp_start, temp_end + 1, temp_step))
        cell = np.array(atoms.get_cell())

        for T in temps:
            # Thermal velocities
            MaxwellBoltzmannDistribution(atoms, temperature_K=T)
            dyn = Langevin(atoms, timestep=dt_fs * units.fs,
                           temperature_K=T, friction=0.01 / units.fs)

            # Equilibrate
            dyn.run(n_equil)

            # Collect fractional-coordinate snapshots
            frac_snaps = []
            for _ in range(n_snapshots):
                dyn.run(collect_interval)
                frac_snaps.append(atoms.get_scaled_positions().copy())

            all_frac = np.array(frac_snaps)  # (n_snapshots, n_atoms, 3)

            # Unwrap PBC crossings (track continuous trajectory)
            for i in range(1, len(all_frac)):
                delta = all_frac[i] - all_frac[i - 1]
                delta -= np.round(delta)          # minimum-image in frac space
                all_frac[i] = all_frac[i - 1] + delta

            # Time-averaged position per atom
            avg_frac = np.mean(all_frac, axis=0)

            # RMS displacement from average -> Lindemann ratio
            delta_frac = all_frac - avg_frac[np.newaxis, :, :]
            delta_cart = np.einsum('ijk,kl->ijl', delta_frac, cell)
            msd = np.mean(np.sum(delta_cart ** 2, axis=2))
            lindemann = np.sqrt(msd) / d_nn

            if lindemann > lindemann_threshold:
                t_C = round(T - 273.15, 1)
                return (t_C,
                        f"Decomposed ~{t_C:.0f} C  (AIMD Lindemann={lindemann:.3f}"
                        f" > {lindemann_threshold} at {T} K)")

        # Survived all temperatures
        t_C = round(max(temps) - 273.15, 1)
        return (t_C,
                f"Stable up to >{t_C:.0f} C  (Lindemann stayed below"
                f" {lindemann_threshold} through {max(temps)} K)")

    except Exception as e:
        return None, f"AIMD failed: {type(e).__name__}: {e}"

In [ ]:
# Cell 9 — CHGNet-based volume change (relaxation)
def volume_change_pct(structure):
    v0 = structure.volume
    try:
        result = relaxer.relax(structure, verbose=False)
        v1 = result['final_structure'].volume
        return round(100.0 * (v1 - v0) / v0, 3), "CHGNet relaxation"
    except Exception as e:
        return None, f"CHGNet relaxation failed: {type(e).__name__}: {e}"
# NOTE: no published result shows CHGNet clearly beats MACE (or vice versa) for this specific task.
# To swap in MACE instead: `pip install mace-torch`, load `mace_mp()`, use it as the ASE calculator
# inside an ASE relaxation (e.g. BFGS) in place of relaxer.relax(), and compare a handful of your
# materials against both before committing to one.

In [ ]:
# Cell 10 — per-material pipeline
def predict_one(cif_path):
    structure = Structure.from_file(cif_path)
    material_id = os.path.splitext(os.path.basename(cif_path))[0]
    out = {'material_id': material_id, 'formula': structure.composition.reduced_formula}

    # Total energy via CHGNet — needed by Ehull and electrochemical-window
    # regardless of which model provides formation energy
    e_tot = chgnet_total_energy(structure)

    cf = crystalformer_predict(structure, cif_path)

    # --- Formation energy: Crystalformer -> CHGNet ---
    if 'formation_energy' in cf:
        out['formation_energy_eV_atom'] = cf['formation_energy']
        out['_src_formation_energy'] = 'Crystalformer'
    else:
        out['formation_energy_eV_atom'] = None
        out['_src_formation_energy'] = 'N/A (Crystalformer not wired up; plug in a real CHGNet formation-energy calc)'

    # --- Band gap: Crystalformer -> ALIGNN -> N/A ---
    bg, bg_src = cf.get('band_gap'), 'Crystalformer' if 'band_gap' in cf else None
    if bg is None:
        bg, err = alignn_predict("mp_gappbe_alignn", cif_path)
        bg_src = 'ALIGNN mp_gappbe_alignn' if bg is not None else f'N/A ({err})'
    out['band_gap_eV'] = bg
    out['_src_band_gap'] = bg_src

    # --- Electronic conductivity: formula on band gap ---
    sigma, cls = electronic_conductivity(bg)
    out['electronic_conductivity_S_cm'], out['electronic_class'] = sigma, cls
    out['_src_conductivity'] = f'Formula (Arrhenius) on band gap [{bg_src}]'

    # --- Bulk/shear modulus: Crystalformer -> ALIGNN -> CHGNet finite-strain ---
    kv, gv, mod_src = cf.get('bulk_modulus'), cf.get('shear_modulus'), None
    # Raw CrystalFormer outputs are log10(GPa); actual values above are 10**log10_value.
    # These log columns are intentionally None when a fallback supplies the moduli.
    out['bulk_modulus_log10_GPa'] = cf.get('bulk_modulus_log10_GPa')
    out['shear_modulus_log10_GPa'] = cf.get('shear_modulus_log10_GPa')
    if 'bulk_modulus' in cf and 'shear_modulus' in cf:
        mod_src = 'Crystalformer'
    else:
        kv_a, kv_err = alignn_predict("jv_bulk_modulus_kv_alignn", cif_path)
        gv_a, gv_err = alignn_predict("jv_shear_modulus_gv_alignn", cif_path)
        if kv_a and gv_a:
            kv, gv, mod_src = kv_a, gv_a, 'ALIGNN jv_bulk/shear_modulus'
        else:
            try:
                from pymatgen.analysis.elasticity import ElasticTensor
                from pymatgen.analysis.elasticity.strain import DeformedStructureSet, Strain
                from pymatgen.analysis.elasticity.stress import Stress
                df_set = DeformedStructureSet(structure, symmetry=False,
                                               norm_strains=[-0.01, 0.01], shear_strains=[-0.01, 0.01])
                strains, stresses = [], []
                for d, defo in zip(df_set, df_set.deformations):
                    p = chgnet_model.predict_structure(d)
                    strains.append(Strain.from_deformation(defo)); stresses.append(Stress(p['s']))
                et = ElasticTensor.from_pseudoinverse(strains, stresses)
                kv, gv = max(2.0, float(et.k_vrh)), max(1.0, float(et.g_vrh))
                mod_src = f'CHGNet finite-strain (ALIGNN: {kv_err or gv_err})'
            except Exception as e:
                kv, gv, mod_src = None, None, f'FAILED ({type(e).__name__}: {e})'
    out['bulk_modulus_GPa'], out['shear_modulus_GPa'], out['_src_moduli'] = kv, gv, mod_src
    # kv and gv are actual GPa values (10**CrystalFormer log10 outputs when CrystalFormer is used).

    # --- Young's modulus, Poisson's ratio, fracture toughness: formulas on the moduli ---
    if kv and gv:
        E, nu = youngs_and_poisson(kv, gv)
        k_ic, gamma_used = fracture_toughness(E, gv, nu)
        out['youngs_modulus_GPa'], out['poisson_ratio'] = E, nu
        out['fracture_toughness_MPa_m05'] = k_ic
        out['_src_youngs_poisson'] = f'Formula on moduli [{mod_src}]'
        out['_src_toughness'] = f'Irwin formula, gamma_s={gamma_used:.2f} J/m2 [{mod_src}]'
    else:
        out['youngs_modulus_GPa'] = out['poisson_ratio'] = out['fracture_toughness_MPa_m05'] = None
        out['_src_youngs_poisson'] = out['_src_toughness'] = 'N/A (no moduli)'

    # --- Ehull (MP phase diagram) ---
    ehull, ehull_src = compute_ehull_mp(structure, e_tot, mpr)
    out['energy_above_hull_eV_atom'], out['_src_ehull'] = ehull, ehull_src

    # --- Thermal decomposition: CHGNet AIMD (Lindemann criterion) ---
    t_decomp, thermal_status = compute_thermal_decomp_aimd(structure)
    out['thermal_decomp_temp_C'], out['thermal_stability'] = t_decomp, thermal_status
    out['_src_thermal_stability'] = 'CHGNet AIMD (Lindemann criterion)'

    # --- Volume change: CHGNet relaxation ---
    vol_pct, vol_src = volume_change_pct(structure)
    out['volume_change_pct'], out['_src_volume_change'] = vol_pct, vol_src

    # --- Electrochemical window: MP grand-potential sweep ---
    v_red, v_ox, window_src = compute_electrochemical_window_mp(structure, e_tot, mpr)
    out['electrochem_window_reduction_V'] = v_red
    out['electrochem_window_oxidation_V'] = v_ox
    out['_src_electrochem_window'] = window_src

    return out

In [ ]:
# Upload CIF files from your computer in Google Colab
from google.colab import files

UPLOAD_DIR = "uploaded_cifs"
os.makedirs(UPLOAD_DIR, exist_ok=True)

uploaded = files.upload()
print(f"Uploaded {len(uploaded)} file(s). Run the next cell to save them and set CIF_DIR.")

Saving selected_50_materials.zip to selected_50_materials (6).zip
Uploaded 1 file(s). Run the next cell to save them and set CIF_DIR.


In [ ]:
import zipfile

saved_paths = []
for fname, content in uploaded.items():
    out_path = os.path.join(UPLOAD_DIR, fname)
    with open(out_path, 'wb') as f:
        f.write(content)

    if fname.lower().endswith('.zip'):
        with zipfile.ZipFile(out_path) as z:
            z.extractall(UPLOAD_DIR)
        print(f"Unzipped {fname} into {UPLOAD_DIR}/")
    else:
        saved_paths.append(out_path)
        print(f"Saved {fname}")

# Update CIF_DIR to point to the correct subdirectory
CIF_DIR = os.path.join(UPLOAD_DIR, 'cif_files')
n_cifs = len([f for f in os.listdir(CIF_DIR) if f.lower().endswith('.cif')])
print(f"\nCIF_DIR set to '{CIF_DIR}' -- {n_cifs} .cif file(s) ready. Now run the batch-run cell below.")

Unzipped selected_50_materials (6).zip into uploaded_cifs/

CIF_DIR set to 'uploaded_cifs/cif_files' -- 50 .cif file(s) ready. Now run the batch-run cell below.


In [ ]:
import os
print(os.listdir(UPLOAD_DIR))

['selected_50_materials (5).zip', 'selected_50_materials (6).zip', 'selected_50_materials (2).zip', 'cif_files', 'selected_50_materials.csv', 'selected_50_materials (1).zip', 'selected_50_materials (4).zip', 'selected_50_materials.zip', 'selected_50_materials (3).zip']


In [ ]:
# Cell 11 — batch run over a folder of CIFs (with per-material timing)
import glob, time

cif_paths = sorted(glob.glob(os.path.join(CIF_DIR, "*.cif")))
total_count = len(cif_paths)
print(f"Found {total_count} CIF files in '{CIF_DIR}'")
print("=" * 70)

rows = []
material_times = []
overall_start = time.time()

for i, p in enumerate(cif_paths):
    mat_name = os.path.splitext(os.path.basename(p))[0]
    mat_start = time.time()
    row = {}
    try:
        row = predict_one(p)
        rows.append(row)
    except Exception as e:
        row = {'material_id': mat_name, '_error': f'{type(e).__name__}: {e}'}
        rows.append(row)
    mat_elapsed = time.time() - mat_start
    material_times.append(mat_elapsed)

    elapsed_total = time.time() - overall_start
    avg_per_mat = elapsed_total / (i + 1)
    eta = avg_per_mat * (total_count - i - 1)
    eta_min, eta_sec = divmod(int(eta), 60)

    # New formatted output per material as requested
    print(f"Material ID: {mat_name} [{i+1}/{total_count}, ETA ~{eta_min}m {eta_sec:02d}s, avg {avg_per_mat:.1f}s/mat]")
    print("Loaded data:")
    for key, value in row.items():
        # Exclude 'material_id', 'formula', and '_src_' fields from the property list
        if key not in ['material_id', 'formula'] and not key.startswith('_src_'):
            print(f"  {key}: {value}")
    print("-" * 70) # Separator for clarity between materials

overall_elapsed = time.time() - overall_start
overall_min, overall_sec = divmod(int(overall_elapsed), 60)

print("=" * 70)
print(f"DONE  {total_count} materials in {overall_min}m {overall_sec:02d}s  "
      f"(avg {overall_elapsed/max(1,total_count):.1f}s per material)")
if material_times:
    print(f"  Fastest: {min(material_times):.1f}s   Slowest: {max(material_times):.1f}s")
print()

df_results = pd.DataFrame(rows)
df_results.to_csv(OUTPUT_CSV, index=False)
print(f"Saved -> {OUTPUT_CSV}  ({len(df_results)} rows x {len(df_results.columns)} cols)")
display(df_results.head())

Found 50 CIF files in 'uploaded_cifs/cif_files'
loaded data: data/_single_predict_tmp/test/raw/raw_data.pkl
dict_keys(['id', 'structure', 'e_form', 'bandgap', 'bulk_modulus', 'shear_modulus', 'ehull'])


Processing...


  0%|          | 0/1 [00:00<?, ?it/s]

Done!


loaded data: data/_single_predict_tmp/test/raw/raw_data.pkl
dict_keys(['id', 'structure', 'e_form', 'bandgap', 'bulk_modulus', 'shear_modulus', 'ehull'])


Processing...


... (990 outputs left out)

  0%|          | 0/1 [00:00<?, ?it/s]

Done!


DEBUG: Crystalformer output for mp-779333: {'formation_energy': -0.8989, 'band_gap': 0.9086, 'bulk_modulus': 2.472, 'shear_modulus': 1.9096, 'e_hull_jarvis': 1.7386}
CHGNet will run on cuda
Material ID: mp-779333 [50/50, ETA ~0m 00s, avg 210.8s/mat]
Loaded data:
  formation_energy_eV_atom: -0.8989
  band_gap_eV: 0.9086
  electronic_conductivity_S_cm: 2.331e-05
  electronic_class: Semiconductor
  bulk_modulus_GPa: 2.472
  shear_modulus_GPa: 1.9096
  youngs_modulus_GPa: 4.56
  poisson_ratio: 0.193
  fracture_toughness_MPa_m05: 0.1
  energy_above_hull_eV_atom: 0.08123483533838005
  thermal_decomp_temp_C: 126.9
  thermal_stability: Decomposed ~127 C  (AIMD Lindemann=0.126 > 0.12 at 400 K)
  volume_change_pct: -0.014
  electrochem_window_reduction_V: None
  electrochem_window_oxidation_V: None
----------------------------------------------------------------------
DONE  50 materials in 175m 41s  (avg 210.8s per material)
  Fastest: 71.9s   Slowest: 738.7s

Saved -> predictions_v2.csv  (50 ro

,material_id,formula,formation_energy_eV_atom,_src_formation_energy,band_gap_eV,_src_band_gap,electronic_conductivity_S_cm,electronic_class,_src_conductivity,bulk_modulus_GPa,...,energy_above_hull_eV_atom,_src_ehull,thermal_decomp_temp_C,thermal_stability,_src_thermal_stability,volume_change_pct,_src_volume_change,electrochem_window_reduction_V,electrochem_window_oxidation_V,_src_electrochem_window
0,mp-1020146,Na2LiB5(PO7)2,-0.5126,Crystalformer,1.3737,Crystalformer,2.888e-09,Semiconductor,Formula (Arrhenius) on band gap [Crystalformer],2.4554,...,0.011478,MP phase diagram (CHGNet energy vs MP competin...,126.9,Decomposed ~127 C (AIMD Lindemann=0.149 > 0.1...,CHGNet AIMD (Lindemann criterion),-0.000,CHGNet relaxation,None,None,Not found stable at any mu_Li in the swept ran...
1,mp-1100505,Li9Mn7O16,-0.9216,Crystalformer,0.7962,Crystalformer,2.050e-04,Semiconductor,Formula (Arrhenius) on band gap [Crystalformer],3.0418,...,0.042513,MP phase diagram (CHGNet energy vs MP competin...,526.9,Decomposed ~527 C (AIMD Lindemann=0.130 > 0.1...,CHGNet AIMD (Lindemann criterion),0.000,CHGNet relaxation,None,None,Not found stable at any mu_Li in the swept ran...
2,mp-1174519,Li5Mn3O8,-0.9484,Crystalformer,1.0487,Crystalformer,1.551e-06,Semiconductor,Formula (Arrhenius) on band gap [Crystalformer],3.0039,...,0.057114,MP phase diagram (CHGNet energy vs MP competin...,526.9,Decomposed ~527 C (AIMD Lindemann=0.124 > 0.1...,CHGNet AIMD (Lindemann criterion),0.000,CHGNet relaxation,None,None,Not found stable at any mu_Li in the swept ran...
3,mp-1175202,Li7Mn4CoO12,-0.9075,Crystalformer,0.9180,Crystalformer,1.943e-05,Semiconductor,Formula (Arrhenius) on band gap [Crystalformer],3.0424,...,0.058006,MP phase diagram (CHGNet energy vs MP competin...,526.9,Decomposed ~527 C (AIMD Lindemann=0.134 > 0.1...,CHGNet AIMD (Lindemann criterion),-0.001,CHGNet relaxation,None,None,Not found stable at any mu_Li in the swept ran...
4,mp-1175325,Li7Mn5O12,-0.9331,Crystalformer,0.9546,Crystalformer,9.574e-06,Semiconductor,Formula (Arrhenius) on band gap [Crystalformer],2.9983,...,0.048179,MP phase diagram (CHGNet energy vs MP competin...,326.9,Decomposed ~327 C (AIMD Lindemann=0.124 > 0.1...,CHGNet AIMD (Lindemann criterion),0.000,CHGNet relaxation,None,None,Not found stable at any mu_Li in the swept ran...


### Action Required: Restart Runtime and Rerun Cells

It appears that the `dgl` library, which ALIGNN relies on, is still not being loaded correctly (`ModuleNotFoundError: No module named 'dgl'`). This often happens when new packages are installed and the Python environment needs to be reset for them to be recognized.

To resolve this, please perform the following steps:

1.  **Restart your Colab runtime:** Go to `Runtime` > `Restart runtime` (or use the shortcut `Ctrl+M .`).
2.  **After the restart, run all cells from Cell 1 (`f2e73ab9`) through Cell 11 (`efcab32e`) sequentially.**

This will ensure that all dependencies are properly installed and all models are initialized in a fresh environment. Once done, we can re-evaluate the prediction output.

In [ ]:
# Cell 12 — which path produced each property (sanity check before trusting results)
for c in [c for c in df_results.columns if c.startswith('_src_')]:
    print(f"\n{c}")
    print(df_results[c].astype(str).str.slice(0, 90).value_counts().to_string())


_src_formation_energy
_src_formation_energy
Crystalformer    50

_src_band_gap
_src_band_gap
Crystalformer    50

_src_conductivity
_src_conductivity
Formula (Arrhenius) on band gap [Crystalformer]    50

_src_moduli
_src_moduli
Crystalformer    50

_src_youngs_poisson
_src_youngs_poisson
Formula on moduli [Crystalformer]    50

_src_toughness
_src_toughness
Irwin formula, gamma_s=0.17 J/m2 [Crystalformer]    12
Irwin formula, gamma_s=0.16 J/m2 [Crystalformer]    11
Irwin formula, gamma_s=0.15 J/m2 [Crystalformer]     8
Irwin formula, gamma_s=0.18 J/m2 [Crystalformer]     5
Irwin formula, gamma_s=0.19 J/m2 [Crystalformer]     4
Irwin formula, gamma_s=0.10 J/m2 [Crystalformer]     3
Irwin formula, gamma_s=0.12 J/m2 [Crystalformer]     3
Irwin formula, gamma_s=0.14 J/m2 [Crystalformer]     2
Irwin formula, gamma_s=0.13 J/m2 [Crystalformer]     2

_src_ehull
_src_ehull
MP phase diagram (CHGNet energy vs MP competing phases)    50

_src_thermal_stability
_src_thermal_stability
CHGNet AIMD

In [ ]:
# Cell 13 — download predictions CSV
# Works in both Google Colab and local Jupyter/VS Code environments

csv_path = OUTPUT_CSV

if not os.path.exists(csv_path):
    print(f"ERROR: {csv_path} not found. Run the batch-run cell (Cell 11) first.")
else:
    file_size = os.path.getsize(csv_path) / 1024
    n_rows = len(pd.read_csv(csv_path))
    print(f"Ready to download: {csv_path}")
    print(f"  Size: {file_size:.1f} KB  |  Rows: {n_rows}")
    print()

    # Try Google Colab download first
    try:
        from google.colab import files
        files.download(csv_path)
        print("Download triggered via Colab.")
    except ImportError:
        # Local environment — show the full path and create a copy on Desktop
        full_path = os.path.abspath(csv_path)
        print(f"Not running on Colab. File is saved locally at:")
        print(f"  {full_path}")
        print()
        # Also copy to Desktop for easy access
        try:
            import shutil
            desktop = os.path.join(os.path.expanduser("~"), "Desktop")
            if os.path.isdir(desktop):
                dest = os.path.join(desktop, os.path.basename(csv_path))
                shutil.copy2(csv_path, dest)
                print(f"Also copied to Desktop:")
                print(f"  {dest}")
        except Exception as e:
            print(f"(Could not copy to Desktop: {e})")
    except Exception as e:
        print(f"Download failed: {type(e).__name__}: {e}")
        print(f"File is at: {os.path.abspath(csv_path)}")

Ready to download: predictions_v2.csv
  Size: 26.6 KB  |  Rows: 50



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered via Colab.
